# 문제 3-1: 콘텐츠분쟁해결 RAG 시스템
콘텐츠분쟁해결 사례집 PDF를 활용한 법률 자문 RAG 시스템을 단계별로 구축합니다.

| 단계 | 내용 |
|---|---|
| 설정 | 패키지 · 상수 · API 키 |
| 0 | 문서 로드 |
| 1 | 문서 분할 (사례 단위) |
| 2 | 임베딩 모델 · 벡터스토어 |
| 3 | 검색기 |
| 4 | LLM |
| 5 | 법률 자문 프롬프트 |
| 6 | LCEL 체인 |
| 7 | 테스트 질문 |
| 8 | 분쟁 유형 분류 (선택) |
| 실행 | 테스트 질문으로 상담 |

### 설정: 패키지 · 상수 · API 키

In [14]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

PDF_PATH = "../data/콘텐츠분쟁해결_사례.pdf"
# 분할 방식(페이지 단위 → 사례 단위)을 바꿨으므로 이전 인덱스와 섞이지 않게 경로를 분리
DB_PATH = "../db/faiss_db_content_dispute_v2"
CHUNK_SIZE = 1500          # 법률 사례는 1200-1800자 권장
CHUNK_OVERLAP = 300        # 사례 맥락 보존을 위해 200-400자
TOP_K = 5                  # 상위 5개 관련 사례 검색
EMBEDDING_MODEL = "solar-embedding-1-large"
CHAT_MODEL = "solar-pro3"
UPSTAGE_BASE_URL = "https://api.upstage.ai/v1"

# 사례 머리 2줄: "■ 1-가-1) 한국소비자원 조정전 상담사례" (분류·기관·유형) + "2006_시스템 오류로 …" (사례 제목)
# 첫 줄의 "1-가-1)"은 분류 번호라 여러 사례가 같으므로, 사례를 구분하는 것은 둘째 줄 제목이다.
CASE_PATTERN = re.compile(r"■\s*([^\n]+)\n\s*([^\n]+)")

load_dotenv()
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(".env 파일에 UPSTAGE_API_KEY 가 없습니다.")
print("설정 완료")

설정 완료


### 0단계: 문서 로드
`PyPDFLoader`는 페이지 1장을 `Document` 1개로 읽습니다.

In [15]:
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

page_lengths = [len(doc.page_content) for doc in documents]
print(f"총 {len(documents)}페이지 로드")
print(f"페이지 길이: 평균 {sum(page_lengths) / len(page_lengths):.0f}자, 최대 {max(page_lengths)}자")

총 109페이지 로드
페이지 길이: 평균 714자, 최대 1776자


### 1단계: 문서 분할 설정

PDF에서 추출한 텍스트는 가이드의 구분자와 표기가 조금 다릅니다.

| 가이드 구분자 | PDF 실제 텍스트 | 대응 |
|---|---|---|
| `【사건개요】` | `【사건개요 】` (닫는 괄호 앞에 공백) | `】`를 빼고 `\n【사건개요`로 |
| `\n■` | `5■ 1-가-1) …` (■ 앞에 페이지 번호) | `\n` 없이 `■` |

또 페이지 평균이 약 700자라서 `split_documents(documents)`로 **페이지마다** 자르면 대부분의 페이지가 그대로 청크 1개가 되고, 페이지 경계에서 사례가 둘로 잘립니다(28건).
그래서 **페이지를 하나로 합친 뒤 사례(■) → 섹션(【…】) 순으로** 자릅니다.

In [16]:
# 페이지를 하나로 합쳐 사례가 페이지 경계에서 잘리지 않게 한다
full_doc = Document(
    page_content="\n".join(doc.page_content for doc in documents),
    metadata={"source": PDF_PATH},
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "■",                 # 사례 경계 (앞에 페이지 번호가 붙어 있어 \n 없이)
        "\n【사건개요",       # 섹션 경계 (PDF 텍스트는 "【사건개요 】"라서 】는 뺀다)
        "\n【쟁점사항",
        "\n【처리경위",
        "\n【처리결과",
        "\n【관련법령",
        "\n\n", "\n", ".", " ", ""
    ],
    keep_separator="start",  # ■가 다음 청크 맨 앞에 붙도록
)
chunks = text_splitter.split_documents([full_doc])

print(f"{len(chunks)}개 청크, 평균 {sum(len(c.page_content) for c in chunks) / len(chunks):.0f}자")
print(f"■로 시작하는 청크: {sum(c.page_content.lstrip().startswith('■') for c in chunks)}개")
for marker in ["【사건개요", "【쟁점사항", "【처리경위", "【처리결과", "【관련법령", "■"]:
    print(f"  {marker}: {full_doc.page_content.count(marker)}회")

105개 청크, 평균 762자
■로 시작하는 청크: 47개
  【사건개요: 56회
  【쟁점사항: 51회
  【처리경위: 56회
  【처리결과: 51회
  【관련법령: 27회
  ■: 56회


#### 1-1. 청크에 사례 제목 붙이기 · 목차 제외
페이지를 합쳤기 때문에 `metadata["page"]`가 없습니다. 대신 청크마다 **사례 제목**을 기록해 출처로 씁니다.
첫 사례(■) 앞부분은 표지·목차입니다. 사례 제목이 모두 들어 있어 검색에 자주 걸리지만 내용이 없으므로 제외합니다.

In [17]:
def attach_case_metadata(chunks):
    """청크마다 사례 제목·분류를 metadata["case"]에 기록한다. ■가 없는 청크는 앞 사례를 이어받는다."""
    current = "사례 머리 이전"
    for chunk in chunks:
        cases = [f"{title.strip()} · {header.strip()}" for header, title in CASE_PATTERN.findall(chunk.page_content)]
        if cases:
            chunk.metadata["case"] = " / ".join(cases)
            current = cases[-1]
        else:
            chunk.metadata["case"] = current
    return chunks


chunks = attach_case_metadata(chunks)

toc_count = sum(c.metadata["case"] == "사례 머리 이전" for c in chunks)
chunks = [c for c in chunks if c.metadata["case"] != "사례 머리 이전"]
print(f"목차 청크 {toc_count}개 제외 → {len(chunks)}개 청크")

print("\n예시:", chunks[0].metadata["case"])
print(chunks[0].page_content[:300])

목차 청크 5개 제외 → 100개 청크

예시: 2006_시스템 오류로 소멸된 아이템 복구 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2006_프로그램 오류로 소멸된 아이템 복구 요구 · 1-가-1) 한국소비자원 조정전 상담사례
■ 1-가-1) 한국소비자원 조정전 상담사례
2006_시스템 오류로 소멸된 아이템 복구 요구
【사건개요 】
신청인은 피신청인이 제공하는 온라인게임서비스를 이용하던 중 시스템 오류로 120,000 원 상당의 
아이템이 소멸되었고 , 이에 피신청인에게 수차례 이의제기하였으나 사과 공지문만 안내 되고 아이템
이 복구 되지 않았다 . 피신청인이 정상적으로 거래하던 중 아이템이 소멸된 것을 확인해 주었으나 
규정상 복구가 불가하다는 답변을 한 것에 대하여 신청인은 정상적으로 이용하다 시스템 오류로 
인해 소멸된 아이템에 대해 원상복구를 요구


### 2단계: 임베딩 모델 설정
청크와 질문을 같은 임베딩 모델로 벡터화해야 비교할 수 있습니다.

In [18]:
embeddings = UpstageEmbeddings(model=EMBEDDING_MODEL)
print(embeddings.model)

solar-embedding-1-large


#### 2-1. 벡터스토어 생성 · 저장
인덱스 파일이 있으면 불러오고, 없으면 임베딩해서 만듭니다. 분할 설정을 바꾸면 `DB_PATH`를 바꾸거나 폴더를 지워야 반영됩니다.

In [19]:
if (Path(DB_PATH) / "index.faiss").exists():
    vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    print(f"기존 인덱스 로드: {DB_PATH}")
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)   # 임베딩 API 호출 (비용 발생)
    vectorstore.save_local(DB_PATH)
    print(f"인덱스 생성·저장: {DB_PATH}")
print(f"벡터 수: {vectorstore.index.ntotal}개")

기존 인덱스 로드: ../db/faiss_db_content_dispute_v2
벡터 수: 100개


### 3단계: 검색기 설정

In [20]:
retriever = vectorstore.as_retriever(
    search_type="similarity",   # 또는 "mmr" (비슷한 사례 중복을 줄이고 다양하게)
    search_kwargs={"k": TOP_K}, # 상위 5개 관련 사례 검색
)

# 검색만 따로 확인
for doc in retriever.invoke("게임 아이템이 사라졌어요"):
    print("-", doc.metadata["case"])

- 2006_시스템 오류로 소멸된 아이템 복구 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2006_프로그램 오류로 소멸된 아이템 복구 요구 · 1-가-1) 한국소비자원 조정전 상담사례
- 2007_인터넷게임서비스 아이템 복구 요구 · 1-가-1) 한국소비자원 조정전 상담사례
- 2008_명의도용으로 개설된 인터넷 게임 계정 삭제 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2008_해킹으로 소실된 아이템 복원 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2008_사용하지 못한 게임머니 구입가의 환급 요구 · 1-가-1) 한국소비자원 조정전 상담사례
- 2009_인터넷 게임 계정 영구이용 제한조치 취소 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2009_사용하지 않았으나 차감된 아이템 배상 요구 · 1-가-1) 한국소비자원 조정전 상담사례
- 2009_명의 도용되어 부당 지급된 게임요금의 환급 요구 · 1-가-1) 한국소비자원 조정전 상담사례 / 2009_해킹으로 손실된 아이템 복구 요청 · 1-가-1) 한국소비자원 조정전 상담사례


### 4단계: LLM 설정

In [21]:
llm = ChatUpstage(
    model=CHAT_MODEL,
    base_url=UPSTAGE_BASE_URL,
    temperature=0.2,
)
print(llm.model_name)

solar-pro3


### 5단계: 법률 자문 프롬프트 작성

In [22]:
prompt_template = """
당신은 콘텐츠 분야 전문 법률 자문가입니다.
아래 분쟁조정 사례들을 바탕으로 정확하고 전문적인 법률 조언을 제공해주세요.

관련 분쟁사례:
{context}

상담 내용: {question}

답변 가이드라인:
1. 제시된 사례들을 근거로 답변하세요
2. 관련 법령이나 조항이 있다면 명시하세요
3. 비슷한 사례의 처리경위와 결과를 참고하여 설명하세요
4. 실무적 해결방안을 단계별로 제시하세요
5. 사례에 없는 내용은 "제시된 사례집에서는 확인할 수 없습니다"라고 명시하세요

전문 법률 조언:"""

# 가이드 6단계처럼 문자열을 체인에 바로 넣으면 오류가 나므로 PromptTemplate으로 감싼다
prompt = PromptTemplate.from_template(prompt_template)
print(prompt.input_variables)

['context', 'question']


### 6단계: LCEL 체인 생성
- `qa_chain`: 가이드의 체인 (답변 문자열만 반환)
- `qa_chain_with_sources`: 검색 1회로 답변과 참고 사례를 함께 반환 `{"docs", "question", "answer"}`

In [23]:
def format_docs(docs):
    """검색된 청크에 [사례 N]과 사례 제목을 붙여 LLM이 구분하기 쉽게 합친다."""
    return "\n\n".join(
        f"[사례 {i}] {doc.metadata.get('case', '')}\n{doc.page_content}"
        for i, doc in enumerate(docs, 1)
    )


def format_sources(docs):
    """참고한 사례 목록 (같은 사례에서 여러 청크가 나오면 한 번만)."""
    cases = list(dict.fromkeys(doc.metadata.get("case", "") for doc in docs))
    return "\n".join(f"  - {case}" for case in cases)


# 가이드의 LCEL 체인
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 출처까지 반환하는 체인
answer_chain = (
    RunnableLambda(lambda x: {"context": format_docs(x["docs"]), "question": x["question"]})
    | prompt
    | llm
    | StrOutputParser()
)
qa_chain_with_sources = RunnableParallel(docs=retriever, question=RunnablePassthrough()).assign(answer=answer_chain)
print("체인 준비 완료")

체인 준비 완료


### 7단계: 테스트 질문 작성

In [24]:
test_questions = [
    "온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?",
    "인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 요구받고 있습니다. 정당한가요?",
    "무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다. 환불 가능한가요?",
    "미성년자가 부모 동의 없이 게임 아이템을 구매했습니다. 환불받을 수 있는 방법이 있나요?",
    "온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해지하고 싶습니다. 가능한가요?",
]
print(f"테스트 질문 {len(test_questions)}개")

테스트 질문 5개


### 8단계: 분쟁 유형 분류 함수 (선택)
가이드 키워드에 테스트 질문의 실제 표현("자동으로 유료", "유료전환", "온라인 교육", "교육 서비스")을 추가했습니다.

In [25]:
DISPUTE_KEYWORDS = {
    "게임": ["게임", "아이템", "계정", "캐릭터", "레벨", "길드"],
    "이러닝": ["강의", "온라인 교육", "온라인교육", "이러닝", "수강", "화상교육", "교육 서비스"],
    "웹콘텐츠": ["웹사이트", "무료체험", "자동결제", "자동으로 유료", "유료전환", "구독", "사이트"],
}


def classify_dispute_type(query):
    query_lower = query.lower()
    for dispute_type, keywords in DISPUTE_KEYWORDS.items():
        if any(keyword in query_lower for keyword in keywords):
            return dispute_type
    return "기타"


for question in test_questions:
    print(f"[{classify_dispute_type(question)}] {question[:30]}...")

[게임] 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게...
[이러닝] 인터넷 강의를 중도 해지하려고 하는데 과도한 위약금을 ...
[웹콘텐츠] 무료체험 후 자동으로 유료전환되어 요금이 청구되었습니다...
[게임] 미성년자가 부모 동의 없이 게임 아이템을 구매했습니다....
[이러닝] 온라인 교육 서비스가 광고와 다르게 제공되어 계약을 해...


### 실행: 테스트 질문으로 법률 상담

In [26]:
for i, question in enumerate(test_questions, 1):
    print("=" * 60)
    print(f"【상담 {i}】 [{classify_dispute_type(question)}] {question}")
    print("-" * 60)
    result = qa_chain_with_sources.invoke(question)
    print(result["answer"])
    print("\n참고 사례:")
    print(format_sources(result["docs"]))

【상담 1】 [게임] 온라인 게임에서 시스템 오류로 아이템이 사라졌는데, 게임회사가 복구를 거부하고 있습니다. 어떻게 해결할 수 있나요?
------------------------------------------------------------
온라인 게임에서 시스템 오류로 아이템이 사라진 경우, 게임회사가 복구를 거부하는 상황에 대한 법적 조언은 다음과 같습니다. 제시된 사례들을 근거로 분석하고, 관련 법령 및 실무적 해결방안을 단계별로 안내드립니다.

---

## 1. **법적 근거 및 관련 법령**

### 가. **민법 제250조 및 제251조 (도품·유실물에 대한 특례)**
- **제250조**: 도품 또는 유실물에 대해 피해자 또는 유실자는 도난 또는 유실한 날로부터 2년 내에 반환을 청구할 수 있음.
- **제251조**: 도품 또는 유실물을 선의로 매수한 경우, 피해자 또는 유실자는 양수인에게 지급한 대가를 변상하고 반환을 청구할 수 있음.

> **적용 가능성**: 시스템 오류로 인한 아이템 소멸이 "도품 또는 유실물"에 해당하는지가 쟁점입니다.  
> **사례 1**에서는 시스템 오류로 인한 아이템 소멸이 인정되었으나, 계정 명의 문제로 복구가 거절된 사례가 있습니다.

### 나. **소비자기본법 제68조**
- **제68조**: 소비자는 분쟁 발생 시 소비자분쟁조정위원회에 조정을 신청할 수 있음.
> **적용 가능성**: 게임회사가 복구를 거부할 경우, 소비자분쟁조정위원회에 조정을 신청할 수 있습니다.

### 다. **약관의 규제에 관한 법률**
- **제6조**: 약관은 소비자에게 부당하게 불리하거나 명확하지 않은 경우 무효가 될 수 있음.
> **적용 가능성**: 게임회사의 약관이 계정 공유나 현금거래를 금지하면서, 시스템 오류에 대한 보상 규정이 없는 경우, 약관의 불공정성을 주장할 수 있습니다.

---

## 2. **유사 사례 분석 및 처리 경위**

### 가. **사례 1: 시스템 오류로 소멸된 아이템 복구 요구 (20